# Colab을 YOLO 원격 추론 서버로 쓰기

이 노트북을 실행해두면, 로컬 `AI_CCTV` 앱이 이 Colab으로 프레임 이미지를 보내서 결함 탐지 결과를 받아옵니다.
로컬 PC는 CPU만 있어도 되고, 실제 계산(YOLO 추론)은 이 Colab의 GPU에서 처리됩니다.

사전 준비:
1. 상단 메뉴 Runtime > Change runtime type 에서 **GPU** 선택
2. https://dashboard.ngrok.com/get-started/your-authtoken 에서 무료 계정 만들고 authtoken 복사 (ngrok 무료 터널을 열려면 필요)
3. 아래 셀을 순서대로 실행 (마지막 셀은 서버가 켜진 채로 계속 실행 중인 상태로 유지됨 - 정지하려면 정지 버튼 클릭)
4. 마지막 셀 실행 후 출력되는 주소를 로컬 AI_CCTV 앱 화면의 "Colab 추론 서버" 입력칸에 붙여넣고 "적용" 클릭

참고:
- 고정 도메인(`NGROK_STATIC_DOMAIN`)을 쓰므로 세션을 껐다 켜도 주소가 안 바뀝니다. 앱에 한 번만 등록하면 됩니다.
- 앱에서 "endpoint ... is offline" (ERR_NGROK_3200) 에러가 보이면 이 노트북의 마지막 셀이 멈춘 것 → 마지막 셀만 다시 실행하면 됩니다.
- 프레임이 아무리 많아도 서버가 8장씩 나눠 GPU에 올리므로 메모리 부족(CUDA OOM)이 나지 않습니다.

In [ ]:
!pip install -q ultralytics flask flask-cors pyngrok

In [ ]:
from pyngrok import ngrok

# https://dashboard.ngrok.com/get-started/your-authtoken 에서 본인 토큰 발급 후 아래에 붙여넣기
NGROK_AUTH_TOKEN = "여기에_본인_ngrok_authtoken_붙여넣기"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# best.pt 업로드 (파일 선택 창이 뜨면 best.pt 선택)
from google.colab import files

uploaded = files.upload()
MODEL_PATH = next(iter(uploaded))
print("model:", MODEL_PATH)

In [ ]:
import traceback

import numpy as np
import cv2
from flask import Flask, request, jsonify
from flask_cors import CORS
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
print("classes:", model.names)

# 한 번에 GPU에 올리는 이미지 수. 리스트를 통째로 predict에 넘기면 전부 한
# 배치로 올라가서 프레임 수가 많을 때 CUDA OOM이 난다 (T4 기준 8~16이 안전).
INFER_BATCH = 8

app = Flask(__name__)
CORS(app)  # 로컬 앱(pywebview/브라우저)에서 이 서버로 직접 요청하려면 CORS 허용 필요


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "classes": model.names})


@app.route("/predict", methods=["POST"])
def predict():
    try:
        conf = float(request.form.get("conf", 0.35))
        imgsz = int(request.form.get("imgsz", 960))
        uploaded_files = request.files.getlist("files")

        images, filenames = [], []
        for f in uploaded_files:
            data = np.frombuffer(f.read(), dtype=np.uint8)
            img = cv2.imdecode(data, cv2.IMREAD_COLOR)
            if img is None:
                continue
            images.append(img)
            filenames.append(f.filename)

        if not images:
            return jsonify({"results": []})

        preds = []
        for i in range(0, len(images), INFER_BATCH):
            preds.extend(model.predict(source=images[i:i + INFER_BATCH], conf=conf, imgsz=imgsz, verbose=False))

        results = []
        for fname, pred in zip(filenames, preds):
            boxes_out = []
            names = getattr(pred, "names", {}) or getattr(model, "names", {}) or {}
            boxes = getattr(pred, "boxes", None)
            if boxes is not None:
                for box in boxes:
                    cls_id = int(box.cls[0])
                    confv = float(box.conf[0])
                    raw_name = str(names.get(cls_id, cls_id))
                    xyxy = [float(v) for v in box.xyxy[0].tolist()]
                    boxes_out.append({
                        "class_id": cls_id,
                        "class_name": raw_name,
                        "confidence": round(confv, 4),
                        "xyxy": xyxy,
                    })
            results.append({"filename": fname, "boxes": boxes_out})

        return jsonify({"results": results})
    except Exception as e:
        # 에러 내용을 JSON으로 돌려줘서 로컬 앱 로그에 실제 원인이 보이게 한다
        traceback.print_exc()
        return jsonify({"error": f"{type(e).__name__}: {e}"}), 500

In [ ]:
# 이 셀을 실행하면 서버가 켜진 채로 계속 돌아갑니다 (정지 버튼 누르기 전까지).
# 고정 도메인을 쓰므로 셀을 껐다 켜도 주소가 바뀌지 않습니다 → 앱에 한 번만 등록하면 됨.
NGROK_STATIC_DOMAIN = "financial-elitism-ebook.ngrok-free.dev"  # ngrok 무료 계정당 1개 제공

ngrok.kill()  # 이전에 열려 있던 터널 정리 (무료 플랜은 동시 세션 1개 제한)
public_url = ngrok.connect(8000, domain=NGROK_STATIC_DOMAIN)
print("=" * 60)
print("이 주소를 로컬 AI_CCTV 앱의 'Colab 추론 서버' 입력칸에 붙여넣으세요:")
print(public_url)
print("=" * 60)

app.run(port=8000)